# 第 54 课：令牌生命周期、刷新轮换与撤销

**把 Agent 的授权放在可信宿主中管理，不把 token 交给模型。**

上一课完成“授权码 + PKCE”。这一课从**已经取得初始凭证**开始，研究访问过期、刷新轮换、并发刷新、撤销以及回执丢失。

每课两个独立文件：本 Notebook 包含全部代码，不需要旁边的 `.py`；完整脚本可以直接在终端运行。

**环境：Python 3.10+，仅标准库；CPU 即可，不下载模型，不连接外部服务。** 实验实际启动两个 `127.0.0.1` HTTP 服务。请在本机或允许回环网络的环境运行。

## 本课的完成标准

学习后，你应该能够解释并验证：

1. Access token 与 refresh token 分别送到哪里，为什么不能互换。
2. 为什么“旧刷新令牌已被消费”不能按一般请求随便重试。
3. 多个工具同时缺少有效凭证时，怎样只刷新一次。
4. 删除本地凭证、撤销访问令牌、撤销刷新家族有什么区别。
5. 为什么撤销成功，不保证所有缓存中的授权立即失效。

建议分两次学习：第一次完成生命周期与并发；第二次完成故障、练习与报告核对。**不需要背诵整个 HTTP 运行器。**

## 先把标准与本课策略分开

参考的是已经发布的 RFC 6749、6750、7009、7662 和 9700。这里不依赖 MCP 新版本细节，也不实现完整 OAuth 服务。

| 层次 | 本课选择 |
|---|---|
| 初次授权 | 由可信 `seed_grant()` 夹具代替，假定上一课已经同意；没有公开签发入口 |
| 令牌类型 | 随机不透明字符串，服务端只用 SHA256 摘要索引；不是 JWT |
| 刷新 | 每次成功都签发新的 access / refresh pair，旧 refresh 失效 |
| 重用检测 | 旧 refresh 再出现，撤销该家族；本课还连带撤销家族内所有 access |
| 显式撤销 | 撤销 refresh 会级联；只撤销 access 不会撤销 refresh |
| 并发 | 单进程锁，不是分布式事务或跨进程凭证库 |
| 时间 | 宿主推进的受控时钟，不等待真实几十秒，也不测延迟 |

**不存在“所有 OAuth 服务都必须按这张表级联”的结论。** 对公共客户端，安全 BCP 要求刷新令牌采用发送方约束或轮换来发现重放；这里选择轮换。[RFC 9700 §4.14](https://www.rfc-editor.org/rfc/rfc9700.html#section-4.14)

In [1]:
import sys
print("Python：", sys.version.split()[0])
print("本课只依赖标准库；运行时不打印 access/refresh token。")

Python： 3.13.5
本课只依赖标准库；运行时不打印 access/refresh token。


## 工程底座：五个可折叠单元

下面内置了完整实现：基础解析 → AS 状态 → RS 校验 → HTTP 传输 → 客户端凭证管理。按顺序运行即可；之后每节用小实验拆开观察。

`Authority` 是授权状态持有者；`LocalServers` 启动两个真实端口，但两者在同一进程内通过可信对象核验令牌。**没有实现网络 introspection endpoint、真实账号、TLS、DPoP 或生产安全认证。**

In [2]:
from __future__ import annotations

import argparse
import copy
import hashlib
import http.client
import json
import math
import platform
import re
import secrets
import socket
import sys
import threading
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlsplit

VERSION = "lesson54-rotation-v1"
CLIENT_ID = "lesson54-public-client"
ALLOWED_SCOPES = frozenset({"docs:read", "calc:use"})
MAX_BODY = 8192
SOURCES = {
    "refresh": "https://www.rfc-editor.org/rfc/rfc6749.html#section-6",
    "security": "https://www.rfc-editor.org/rfc/rfc9700.html#section-4.14",
    "revocation": "https://www.rfc-editor.org/rfc/rfc7009.html",
    "introspection": "https://www.rfc-editor.org/rfc/rfc7662.html#section-4",
    "bearer": "https://www.rfc-editor.org/rfc/rfc6750.html#section-3.1",
    "http": "https://docs.python.org/3/library/http.server.html",
}

class LessonError(ValueError):
    pass

class OAuthError(Exception):
    def __init__(self, error: str, status: int = 400):
        super().__init__(error)  # 不把输入或凭证写进异常
        self.error, self.status = error, status

class TransportUnknown(Exception):
    """响应未得到确认：服务可能已经提交，不能直接重发旧 refresh token。"""

class ReauthorizationRequired(Exception):
    pass


def require(condition: bool, message: str) -> None:
    if not condition:
        raise LessonError(message)


def canonical(value) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def token_digest(token: str) -> str:
    require(isinstance(token, str) and 1 <= len(token) <= 1024,
            "token 必须是非空有界字符串")
    return hashlib.sha256(token.encode("utf-8")).hexdigest()


def strict_json(text: str):
    def pairs(items):
        out = {}
        for key, value in items:
            require(key not in out, "重复 JSON 键")
            out[key] = value
        return out
    def bad_constant(_):
        raise LessonError("非有限 JSON 数值")
    return json.loads(text, object_pairs_hook=pairs, parse_constant=bad_constant)


def strict_form(text: str) -> dict[str, str]:
    require(len(text.encode("utf-8")) <= MAX_BODY, "表单过长")
    require(not re.search(r"%(?![0-9a-fA-F]{2})", text), "错误百分号编码")
    out = {}
    for key, value in parse_qsl(text, keep_blank_values=True, strict_parsing=True,
                                encoding="utf-8", errors="strict", max_num_fields=16):
        require(key not in out, "重复表单字段")
        out[key] = value
    return out


def parse_scope(text: str) -> frozenset[str]:
    require(isinstance(text, str) and 0 < len(text) <= 128, "scope 类型或长度错误")
    parts = text.split(" ")
    require(all(parts) and len(parts) == len(set(parts)), "scope 不能为空或重复")
    require(set(parts) <= ALLOWED_SCOPES, "未知 scope")
    return frozenset(parts)


class ManualClock:
    """实验时钟，不是服务性能时间。没有 HTTP 调整入口；只有可信宿主推进它。"""
    def __init__(self, value: float = 1_700_000_000.0):
        require(isinstance(value, (int, float)) and not isinstance(value, bool)
                and math.isfinite(value), "时间错误")
        self.value, self.lock = float(value), threading.RLock()
    def now(self) -> float:
        with self.lock:
            return self.value
    def advance(self, seconds: float) -> None:
        require(isinstance(seconds, (int, float)) and not isinstance(seconds, bool)
                and math.isfinite(seconds) and seconds >= 0, "时间不能倒退或非有限")
        with self.lock:
            self.value += seconds


@dataclass(frozen=True)
class Policy:
    access_ttl: int = 10
    refresh_idle: int = 40
    family_absolute: int = 120
    def __post_init__(self):
        require(all(type(x) is int and x > 0 for x in
                    (self.access_ttl, self.refresh_idle, self.family_absolute)),
                "有效期必须是正整数")

In [3]:
@dataclass
class Family:
    client_id: str
    subject: str
    audience: str
    grant_scopes: frozenset[str]
    absolute_exp: float
    idle_exp: float
    generation: int = 0
    revoked: bool = False
    revoke_reason: str = ""

@dataclass
class RefreshRecord:
    family_id: str
    generation: int
    used: bool = False

@dataclass
class AccessRecord:
    family_id: str
    scopes: frozenset[str]
    exp: float
    revoked: bool = False

@dataclass(frozen=True)
class TokenPair:
    # 避免在 Notebook 中不慎展示对象时打印凭证。
    access_token: str = field(repr=False)
    refresh_token: str = field(repr=False)
    expires_at: float
    scopes: frozenset[str]


class Authority:
    """单进程、内存 AS。锁覆盖验证、消费旧 RT 与签发下一代的整个临界区。

    课堂策略：旧 RT 重用/显式撤销 RT -> 撤销整个家族和所有 AT。
    普通轮换不撤销旧 AT；只撤销 AT 不撤销 RT。
    refresh 空闲/绝对到期不自动撤销仍未到期的 AT。
    这些级联选择不是所有服务统一的标准行为。
    """
    def __init__(self, clock: ManualClock | None = None, policy: Policy | None = None):
        self.clock = clock or ManualClock()
        self.policy = policy or Policy()
        self.issuer = "https://issuer.lesson54.invalid"
        self.resource = "https://resource.lesson54.invalid/tools"
        self.lock = threading.RLock()
        self.families: dict[str, Family] = {}
        self.refreshes: dict[str, RefreshRecord] = {}
        self.accesses: dict[str, AccessRecord] = {}
        self.events: list[dict] = []
        self.stats = {"refresh_attempts": 0, "refresh_successes": 0,
                      "reuse_detected": 0, "revocation_requests": 0}
        self.max_records = 10000  # 课堂内存边界；不静默删除旧 RT 的重用线索

    def _capacity(self):
        if len(self.refreshes) >= self.max_records or len(self.accesses) >= self.max_records:
            raise OAuthError("temporarily_unavailable", 503)

    def _mint(self, fid: str, scopes: frozenset[str]) -> dict:
        family = self.families[fid]
        at, rt = secrets.token_urlsafe(32), secrets.token_urlsafe(32)
        self.accesses[token_digest(at)] = AccessRecord(
            fid, scopes, self.clock.now() + self.policy.access_ttl)
        self.refreshes[token_digest(rt)] = RefreshRecord(fid, family.generation)
        return {"access_token": at, "refresh_token": rt, "token_type": "Bearer",
                "expires_in": self.policy.access_ttl, "scope": " ".join(sorted(scopes))}

    def seed_grant(self, scopes: str = "docs:read calc:use", client_id: str = CLIENT_ID,
                   subject: str = "simulated-alice") -> dict:
        """可信教学夹具：假定上一课已经完成批准。不是 HTTP 发 token 接口。"""
        parsed = parse_scope(scopes)
        require(client_id == CLIENT_ID, "课堂只登记一个 public client")
        with self.lock:
            self._capacity()
            fid = "F" + str(len(self.families) + 1)
            now = self.clock.now()
            self.families[fid] = Family(client_id, subject, self.resource, parsed,
                now + self.policy.family_absolute,
                min(now + self.policy.refresh_idle, now + self.policy.family_absolute))
            self.events.append({"event": "grant_fixture", "family": fid})
            return self._mint(fid, parsed)

    def _revoke_family(self, fid: str, reason: str):
        family = self.families[fid]
        family.revoked, family.revoke_reason = True, reason
        self.events.append({"event": "family_revoked", "family": fid, "reason": reason})

    def refresh(self, token: str, client_id: str, scope: str | None = None,
                resource: str | None = None) -> dict:
        digest = token_digest(token)
        with self.lock:
            self.stats["refresh_attempts"] += 1
            old = self.refreshes.get(digest)
            if old is None:
                raise OAuthError("invalid_grant")
            family = self.families[old.family_id]
            # client_id 只表示 public-client 绑定，不是客户端密码认证。
            if family.client_id != client_id:
                raise OAuthError("invalid_grant")
            if family.revoked:
                raise OAuthError("invalid_grant")
            # 先识别历史重用，之后才判断当前代是否到期。
            if old.used:
                self.stats["reuse_detected"] += 1
                self._revoke_family(old.family_id, "refresh_reuse")
                raise OAuthError("invalid_grant")
            now = self.clock.now()
            if now >= family.absolute_exp or now >= family.idle_exp:
                raise OAuthError("invalid_grant")
            if resource is not None and resource != family.audience:
                raise OAuthError("invalid_target")
            try:
                requested = family.grant_scopes if scope is None else parse_scope(scope)
            except LessonError:
                raise OAuthError("invalid_scope") from None
            if not requested <= family.grant_scopes:
                raise OAuthError("invalid_scope")
            self._capacity()  # 容量拒绝发生在消费旧 RT 之前
            old.used = True
            family.generation += 1
            family.idle_exp = min(now + self.policy.refresh_idle, family.absolute_exp)
            # 新 AT 可缩小 scope，但轮换的 RT 仍承接原 grant scope（RFC6749 §6）。
            result = self._mint(old.family_id, requested)
            self.stats["refresh_successes"] += 1
            self.events.append({"event": "rotated", "family": old.family_id,
                                "generation": family.generation})
            return result

    def inspect_access(self, token: str, audience: str) -> dict:
        """可信 RS 内部调用；形状类似 introspection，不暴露公开 introspection 端点。"""
        try:
            digest = token_digest(token)
        except LessonError:
            return {"active": False}
        with self.lock:
            record = self.accesses.get(digest)
            if record is None:
                return {"active": False}
            family = self.families[record.family_id]
            if (record.revoked or family.revoked or self.clock.now() >= record.exp
                    or family.audience != audience):
                return {"active": False}
            return {"active": True, "sub": family.subject, "aud": family.audience,
                    "iss": self.issuer, "scope": " ".join(sorted(record.scopes)),
                    "exp": record.exp}

    def revoke(self, token: str, client_id: str, hint: str | None = None) -> None:
        """不存在的 token 仍成功；hint 不是可信类型标记，按实际表查找。
        public client 的 ID 绑定不构成身份认证。真实 confidential client 应另行认证。
        """
        digest = token_digest(token)
        with self.lock:
            self.stats["revocation_requests"] += 1
            rt, at = self.refreshes.get(digest), self.accesses.get(digest)
            if rt is not None:
                family = self.families[rt.family_id]
                if family.client_id == client_id:
                    self._revoke_family(rt.family_id, "explicit_refresh_revocation")
            elif at is not None:
                family = self.families[at.family_id]
                if family.client_id == client_id:
                    at.revoked = True
                    self.events.append({"event": "access_revoked", "family": at.family_id})

    def family_status(self, refresh_token: str) -> dict:
        """仅用于可信课堂观察，不暴露给网络，也不导出 token 摘要。"""
        with self.lock:
            r = self.refreshes[token_digest(refresh_token)]
            f = self.families[r.family_id]
            return {"family": r.family_id, "generation": f.generation,
                    "revoked": f.revoked, "reason": f.revoke_reason,
                    "refresh_used": r.used,
                    "absolute_remaining": max(0, f.absolute_exp - self.clock.now()),
                    "idle_remaining": max(0, f.idle_exp - self.clock.now())}

In [4]:
class ResourceGuard:
    def __init__(self, authority: Authority, cache_ttl: float = 0):
        require(type(cache_ttl) in (int, float) and math.isfinite(cache_ttl)
                and cache_ttl >= 0, "cache_ttl 非法")
        self.authority, self.cache_ttl = authority, cache_ttl
        self.cache: dict[str, tuple[float, dict]] = {}
        self.lock = threading.RLock()

    def authorize(self, token: str, scope: str) -> dict:
        key, now = token_digest(token), self.authority.clock.now()
        with self.lock:
            cached = self.cache.get(key)
            if cached is not None and now < cached[0]:
                metadata = copy.deepcopy(cached[1])
            else:
                metadata = self.authority.inspect_access(token, self.authority.resource)
                if metadata["active"] and self.cache_ttl > 0:
                    if len(self.cache) >= 1024:
                        self.cache.clear()  # 放弃缓存是安全退化，不回避权威验证
                    expiry = min(now + self.cache_ttl, metadata["exp"])
                    self.cache[key] = (expiry, copy.deepcopy(metadata))
        if not metadata["active"]:
            raise OAuthError("invalid_token", 401)
        if scope not in metadata["scope"].split():
            raise OAuthError("insufficient_scope", 403)
        return metadata

In [5]:
class LocalServers:
    """AS 提供 /token、/revoke；RS 提供 /docs、/calculate、/cached/docs。
    AS/RS 不同监听端口，同进程共享可信验证对象；并非独立信任域。
    loss-arm 仅是宿主测试方法，不是 HTTP 功能或客户端可请求的头。
    """
    def __init__(self, authority: Authority | None = None):
        self.authority = authority or Authority()
        self.live = ResourceGuard(self.authority, 0)
        self.cached = ResourceGuard(self.authority, 5)
        self.trace, self.trace_lock = [], threading.Lock()
        self.loss_next_refresh = False
        self.loss_lock = threading.Lock()
        self.executions = {"docs": 0, "calculate": 0}
        self.servers, self.threads = [], []
        self.as_origin = self.rs_origin = ""

    def arm_response_loss(self):
        with self.loss_lock:
            self.loss_next_refresh = True

    def __enter__(self):
        owner = self
        class Handler(BaseHTTPRequestHandler):
            protocol_version = "HTTP/1.1"
            def log_message(self, *_):
                pass
            def setup(self):
                super().setup()
                self.connection.settimeout(3)
            def record(self, status):
                with owner.trace_lock:
                    # 不记录 URL 查询、headers、form、响应正文或 token 摘要。
                    owner.trace.append({"server": self.server.role,
                                        "method": self.command,
                                        "path": urlsplit(self.path).path, "status": status})
            def send_result(self, status, data=None):
                body = b"" if data is None else canonical(data).encode("utf-8")
                self.send_response(status)
                self.send_header("Content-Length", str(len(body)))
                self.send_header("Connection", "close")
                self.send_header("Cache-Control", "no-store")
                self.send_header("Pragma", "no-cache")
                if data is not None:
                    self.send_header("Content-Type", "application/json; charset=utf-8")
                if self.server.role == "rs" and status in (401, 403):
                    self.send_header("WWW-Authenticate", 'Bearer error="' + data["error"] + '"')
                self.end_headers()
                try:
                    if body:
                        self.wfile.write(body)
                except (BrokenPipeError, ConnectionResetError):
                    pass
                self.close_connection = True
                self.record(status)
            def validate_transport(self):
                expected = f"127.0.0.1:{self.server.server_port}"
                if self.headers.get("Host") != expected or self.headers.get("Origin"):
                    raise OAuthError("forbidden_origin", 403)
                if urlsplit(self.path).query:
                    raise OAuthError("invalid_request")
                if len(self.headers.get_all("Authorization", [])) > 1:
                    raise OAuthError("invalid_request")
            def read_form(self):
                if self.headers.get("Transfer-Encoding"):
                    raise OAuthError("invalid_request")
                lengths = self.headers.get_all("Content-Length", [])
                if len(lengths) != 1 or not lengths[0].isdigit():
                    raise OAuthError("invalid_request")
                size = int(lengths[0])
                if not 0 < size <= MAX_BODY:
                    raise OAuthError("invalid_request")
                media = self.headers.get("Content-Type", "").split(";", 1)[0].strip()
                if media != "application/x-www-form-urlencoded":
                    raise OAuthError("invalid_request", 415)
                data = self.rfile.read(size)
                if len(data) != size:
                    raise OAuthError("invalid_request")
                return strict_form(data.decode("utf-8", errors="strict"))
            def do_POST(self):
                try:
                    self.validate_transport()
                    if self.server.role != "as":
                        raise OAuthError("not_found", 404)
                    form = self.read_form()
                    if self.path == "/token":
                        if form.get("grant_type") != "refresh_token":
                            raise OAuthError("unsupported_grant_type")
                        if not form.get("refresh_token") or not form.get("client_id"):
                            raise OAuthError("invalid_request")
                        result = owner.authority.refresh(form["refresh_token"], form["client_id"],
                            form.get("scope") or None, form.get("resource") or None)
                        # 未识别字段按 OAuth 规则忽略；不能影响权限判断。
                        with owner.loss_lock:
                            drop, owner.loss_next_refresh = owner.loss_next_refresh, False
                        if drop:
                            self.record("committed_response_dropped")
                            self.close_connection = True
                            try:
                                self.connection.shutdown(socket.SHUT_RDWR)
                            except OSError:
                                pass
                            self.connection.close()
                            return
                        self.send_result(200, result)
                    elif self.path == "/revoke":
                        if not form.get("token") or not form.get("client_id"):
                            raise OAuthError("invalid_request")
                        owner.authority.revoke(form["token"], form["client_id"], form.get("token_type_hint"))
                        self.send_result(200)  # 合法已撤销/未知 token 都空正文 200
                    else:
                        raise OAuthError("not_found", 404)
                except OAuthError as e:
                    self.send_result(e.status, {"error": e.error})
                except (LessonError, ValueError, UnicodeError, TimeoutError):
                    self.send_result(400, {"error": "invalid_request"})
            def do_GET(self):
                try:
                    self.validate_transport()
                    if self.server.role != "rs" or self.path not in ("/docs", "/calculate", "/cached/docs"):
                        raise OAuthError("not_found", 404)
                    authorization = self.headers.get("Authorization", "")
                    if not authorization.startswith("Bearer ") or not authorization[7:]:
                        raise OAuthError("invalid_token", 401)
                    scope = "calc:use" if self.path == "/calculate" else "docs:read"
                    guard = owner.cached if self.path == "/cached/docs" else owner.live
                    metadata = guard.authorize(authorization[7:], scope)
                    kind = "calculate" if scope == "calc:use" else "docs"
                    with owner.trace_lock:
                        owner.executions[kind] += 1
                    data = {"value": 16} if kind == "calculate" else {"text": "教学资料：自动备份保留14天。"}
                    self.send_result(200, {"subject": metadata["sub"], "data": data})
                except OAuthError as e:
                    self.send_result(e.status, {"error": e.error})
                except LessonError:
                    self.send_result(400, {"error": "invalid_request"})
        for role in ("as", "rs"):
            server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
            server.daemon_threads = True
            server.role = role
            self.servers.append(server)
        self.as_origin = f"http://127.0.0.1:{self.servers[0].server_port}"
        self.rs_origin = f"http://127.0.0.1:{self.servers[1].server_port}"
        self.authority.issuer = self.as_origin
        self.authority.resource = self.rs_origin + "/tools"
        for server in self.servers:
            thread = threading.Thread(target=server.serve_forever,
                                      kwargs={"poll_interval": 0.02}, daemon=True)
            self.threads.append(thread)
            thread.start()
        return self

    def __exit__(self, *_):
        for server in self.servers:
            server.shutdown()
            server.server_close()
        for thread in self.threads:
            thread.join(timeout=3)
            require(not thread.is_alive(), "服务线程未退出")


def request(origin: str, method: str, path: str, form: dict | None = None,
            bearer: str | None = None, body_override: bytes | None = None,
            extra_headers: dict | None = None) -> tuple[int, dict, dict]:
    """只连接显式回环地址；不跟随重定向；传输异常不会自动重试。"""
    url = urlsplit(origin)
    require(url.scheme == "http" and url.hostname == "127.0.0.1" and url.port
            and not url.username and not url.password and not url.query and not url.fragment
            and url.path in ("", "/"), "仅允许可信的本地 origin")
    require(path.startswith("/") and "?" not in path, "不把凭证放入 URL")
    headers = {"Accept": "application/json"}
    if bearer is not None:
        headers["Authorization"] = "Bearer " + bearer
    body = None
    if form is not None:
        body = urlencode(form).encode("utf-8")
        headers["Content-Type"] = "application/x-www-form-urlencoded"
    if body_override is not None:
        body = body_override
        headers["Content-Type"] = "application/x-www-form-urlencoded"
    if extra_headers:
        headers.update(extra_headers)
    connection = http.client.HTTPConnection(url.hostname, url.port, timeout=3)
    try:
        connection.request(method, path, body=body, headers=headers)
        response = connection.getresponse()
        data = response.read(MAX_BODY + 1)
        if len(data) > MAX_BODY or 300 <= response.status < 400:
            raise TransportUnknown("响应不可靠；没有重试")
        decoded = {} if not data else strict_json(data.decode("utf-8"))
        return response.status, decoded, {k.lower(): v for k, v in response.getheaders()}
    except (OSError, http.client.HTTPException, LessonError, ValueError) as e:
        raise TransportUnknown("传输或响应解析失败；刷新结果未知") from None
    finally:
        connection.close()


def refresh_http(servers: LocalServers, token: str, scope: str | None = None,
                 client_id: str = CLIENT_ID, resource: str | None = None):
    form = {"grant_type": "refresh_token", "refresh_token": token, "client_id": client_id}
    if scope is not None:
        form["scope"] = scope
    if resource is not None:
        form["resource"] = resource
    return request(servers.as_origin, "POST", "/token", form=form)


def revoke_http(servers: LocalServers, token: str, hint: str = "refresh_token"):
    return request(servers.as_origin, "POST", "/revoke", form={
        "token": token, "token_type_hint": hint, "client_id": CLIENT_ID})

In [6]:
class TokenManager:
    """单进程单实例的 single-flight；锁不是跨进程锁。
    token 只放在可信宿主内存，不传给模型或写入课程报告。
    """
    def __init__(self, servers: LocalServers, issued: dict, leeway: float = 1.0):
        require(type(leeway) in (int, float) and math.isfinite(leeway) and leeway >= 0,
                "leeway 非法")
        self.servers, self.clock = servers, servers.authority.clock
        self.leeway, self.lock = leeway, threading.RLock()
        self.state = "ready"
        self.refresh_requests = 0
        self.pair: TokenPair | None = self._decode_pair(issued, self.clock.now())
        require(leeway < issued["expires_in"], "leeway 必须小于 access TTL")

    def _decode_pair(self, response: dict, request_started: float) -> TokenPair:
        require(isinstance(response, dict) and response.get("token_type", "").lower() == "bearer",
                "token 响应类型错误")
        require(type(response.get("expires_in")) is int and response["expires_in"] > 0,
                "expires_in 错误")
        token_digest(response.get("access_token"))
        token_digest(response.get("refresh_token"))
        return TokenPair(response["access_token"], response["refresh_token"],
                         request_started + response["expires_in"], parse_scope(response["scope"]))

    def get_access_token(self) -> str:
        # 锁必须覆盖到期复查、HTTP 刷新和整对替换。
        with self.lock:
            if self.state != "ready" or self.pair is None:
                raise ReauthorizationRequired(self.state)
            if self.clock.now() < self.pair.expires_at - self.leeway:
                return self.pair.access_token
            self.state = "refreshing"
            old = self.pair
            started = self.clock.now()
            self.refresh_requests += 1
            try:
                status, body, _ = refresh_http(self.servers, old.refresh_token)
                if status != 200:
                    self.state = "reauth_required" if body.get("error") == "invalid_grant" else "refresh_failed"
                    raise ReauthorizationRequired(self.state)
                replacement = self._decode_pair(body, started)
            except TransportUnknown:
                self.state = "refresh_unknown"
                raise ReauthorizationRequired(self.state) from None
            except LessonError:
                self.state = "refresh_unknown"  # 服务器可能已轮换，不能使用半份响应
                raise ReauthorizationRequired(self.state) from None
            self.pair = replacement  # 一次性替换，不产生“新 AT + 旧 RT”组合
            self.state = "ready"
            return replacement.access_token

    def call_readonly(self, path: str = "/docs") -> dict:
        token = self.get_access_token()
        status, body, _ = request(self.servers.rs_origin, "GET", path, bearer=token)
        # 403 不是刷新理由；401 也不被循环吞掉。本课要求重新核对授权。
        if status == 401:
            with self.lock:
                if self.pair is not None and self.pair.access_token == token:
                    self.state = "reauth_required"
        return {"status": status, "error": body.get("error"), "body": body}

    def logout_local(self):
        with self.lock:
            self.pair = None
            self.state = "logged_out_local"

    def revoke_and_forget(self) -> str:
        with self.lock:
            if self.pair is None:
                return "no_local_credentials"
            status_text = "revocation_unknown"
            try:
                status, _, _ = revoke_http(self.servers, self.pair.refresh_token)
                if status == 200:
                    status_text = "revocation_confirmed"
            except TransportUnknown:
                pass
            finally:
                self.pair = None
                self.state = status_text
            return status_text

    def summary(self) -> dict:
        with self.lock:
            return {"state": self.state, "refresh_requests": self.refresh_requests,
                    "has_local_credentials": self.pair is not None,
                    "access_seconds_left": None if self.pair is None else
                    max(0, self.pair.expires_at - self.clock.now())}

## 1．两类凭证不能互换

- **Access token** 发送给资源服务，用来读取资料或调用工具。
- **Refresh token** 发送给授权服务器的 token endpoint，用于取得新的访问凭证，不给资料工具、不放进模型上下文。

`client_id` 对公共客户端只是公开标识，**不是密码，也不证明请求方身份**。

服务端持有摘要不等于客户端不用保存秘密：客户端仍需保管原始随机串。本课只将它们保存在宿主内存里，不提供持久秘密存储。

[RFC 6749 §1.5 / §6](https://www.rfc-editor.org/rfc/rfc6749.html#section-6)

In [7]:
authority = Authority()
issued = authority.seed_grant("docs:read calc:use")
print("响应字段：", sorted(issued))
print("access 被资源服务接受：", authority.inspect_access(issued["access_token"], authority.resource)["active"])
print("refresh 被当作 access：", authority.inspect_access(issued["refresh_token"], authority.resource)["active"])
# 不显示 issued；里面有本次有效凭证。

响应字段： ['access_token', 'expires_in', 'refresh_token', 'scope', 'token_type']
access 被资源服务接受： True
refresh 被当作 access： False


## 2．过期边界：`now == exp` 已经过期

默认 access 有效期是 **10 秒**。这是为了教学，不是生产环境推荐值。

刷新资格使用两个边界：最后一次成功刷新后最多空闲 **40 秒**；整个家族最长 **120 秒**。后续成功刷新可以更新空闲期限，但不能延长绝对期限。

一般形式是：

```python
refresh_allowed = now < min(family_absolute_exp, idle_exp)
```

这些具体期限是本课策略。令牌撤销与自然到期也不同：一个刷新凭证不能再使用，并不必然要求其所有尚未到期的 access 立即失效。

In [8]:
authority = Authority()
pair = authority.seed_grant()
observations = []
for delta in [0, 9, 1]:
    authority.clock.advance(delta)
    observations.append(authority.inspect_access(pair["access_token"], authority.resource)["active"])
print("第0、9、10秒的access状态：", observations)
renewed = authority.refresh(pair["refresh_token"], CLIENT_ID)
print("第10秒依然可以用refresh换取新access：", authority.inspect_access(renewed["access_token"], authority.resource)["active"])

第0、9、10秒的access状态： [True, True, False]
第10秒依然可以用refresh换取新access： True


## 3．刷新轮换：一个家族，不是互不相关的随机串

正常路径：

```text
RT₀ → AT₁ + RT₁
         RT₁ → AT₂ + RT₂
```

服务器仍然保留 RT₀、RT₁ 曾经属于哪个家族的线索。再次看到已消费的 RT₀，才能识别旧令牌重用并使后续凭证失效。

**服务器无法仅靠重用发现来判断是攻击者还是合法客户端的重试。** 本课没有宽限窗口，也不重复返回上次的完整刷新响应。

正常轮换 RT 不会立即撤销旧 AT；旧 AT 按自身有效期或撤销规则处理。重用检测则是本课明确的家族级撤销事件。

[RFC 9700 §4.14.2](https://www.rfc-editor.org/rfc/rfc9700.html#section-4.14.2)

In [9]:
authority = Authority()
old = authority.seed_grant()
new = authority.refresh(old["refresh_token"], CLIENT_ID)
print("刷新令牌已变化：", old["refresh_token"] != new["refresh_token"])
print("旧access暂时仍有效：", authority.inspect_access(old["access_token"], authority.resource)["active"])
try:
    authority.refresh(old["refresh_token"], CLIENT_ID)
except OAuthError as error:
    print("故意重复使用旧refresh：", error.error)
print("重用后新access仍有效：", authority.inspect_access(new["access_token"], authority.resource)["active"])
print("家族状态：", authority.family_status(new["refresh_token"]))

刷新令牌已变化： True
旧access暂时仍有效： True
故意重复使用旧refresh： invalid_grant
重用后新access仍有效： False
家族状态： {'family': 'F1', 'generation': 1, 'revoked': True, 'reason': 'refresh_reuse', 'refresh_used': False, 'absolute_remaining': 120.0, 'idle_remaining': 40.0}


## 4．Scope：刷新不能增加原本没批准的权限

RFC 6749 §6 允许刷新时请求更小的 access scope；但新签发的 refresh token 仍对应原 refresh 的授权范围。

因此：原始授权有 `docs:read calc:use`，本次只申请 `docs:read`，**不意味着以后永远不能再申请原有的 calc:use**。这不是越权，而是仍在原批准范围内。

相反，原授权只有 `docs:read`，不能通过刷新把它升级成 `calc:use`。本课也拒绝更换目标 resource。

下面同时观察刷新 scope 与绝对期限。

In [10]:
def demo_scope_and_expiry() -> dict:
    with LocalServers() as servers:
        issued = servers.authority.seed_grant("docs:read calc:use")
        status, narrowed, _ = refresh_http(servers, issued["refresh_token"], "docs:read")
        read, _, _ = request(servers.rs_origin, "GET", "/docs", bearer=narrowed["access_token"])
        calc, _, _ = request(servers.rs_origin, "GET", "/calculate", bearer=narrowed["access_token"])
        _, restored, _ = refresh_http(servers, narrowed["refresh_token"])
        calc_restored, _, _ = request(servers.rs_origin, "GET", "/calculate", bearer=restored["access_token"])
        only_read = servers.authority.seed_grant("docs:read")
        extra, body, _ = refresh_http(servers, only_read["refresh_token"], "calc:use")
        target, target_body, _ = refresh_http(servers, only_read["refresh_token"], resource="https://other.invalid")
        scope_result = {"narrow_refresh_status": status, "narrow_read": read, "narrow_calc": calc,
                        "restore_original_scope_calc": calc_restored,
                        "beyond_grant_status": extra, "beyond_grant_error": body["error"],
                        "wrong_resource_status": target, "wrong_resource_error": target_body["error"]}
    authority = Authority(policy=Policy(access_ttl=10, refresh_idle=20, family_absolute=45))
    current = authority.seed_grant()
    deadlines = [authority.family_status(current["refresh_token"])["absolute_remaining"]]
    for _ in range(3):
        authority.clock.advance(14)
        current = authority.refresh(current["refresh_token"], CLIENT_ID)
        deadlines.append(authority.family_status(current["refresh_token"])["absolute_remaining"])
    authority.clock.advance(3)
    try:
        authority.refresh(current["refresh_token"], CLIENT_ID)
        error = "unexpected_success"
    except OAuthError as e:
        error = e.error
    return {"scopes": scope_result, "absolute_remaining_after_rotations": deadlines,
            "at_absolute_boundary": error,
            "access_can_outlive_refresh_eligibility": authority.inspect_access(
                current["access_token"], authority.resource)["active"]}

In [11]:
scope_expiry_report = demo_scope_and_expiry()
print(json.dumps(scope_expiry_report, ensure_ascii=False, indent=2))

{
  "scopes": {
    "narrow_refresh_status": 200,
    "narrow_read": 200,
    "narrow_calc": 403,
    "restore_original_scope_calc": 200,
    "beyond_grant_status": 400,
    "beyond_grant_error": "invalid_scope",
    "wrong_resource_status": 400,
    "wrong_resource_error": "invalid_target"
  },
  "absolute_remaining_after_rotations": [
    45.0,
    31.0,
    17.0,
    3.0
  ],
  "at_absolute_boundary": "invalid_grant",
  "access_can_outlive_refresh_eligibility": true
}


输出中的绝对剩余时间应该是 `[45,31,17,3]`，不会每次刷新又变回45秒。

本例在第42秒签发的 AT 到第52秒才过期；第45秒刷新资格已经结束，但 AT 仍可能有效。**refresh 不可用与 access 不可用不是同一状态。**

客户端遇到 `403 insufficient_scope` 不应该循环刷新。它需要检查宿主权限或取得额外授权；新 token 本身不会创造未批准的权限。[RFC 6750 §3.1](https://www.rfc-editor.org/rfc/rfc6750.html#section-3.1)

In [12]:
authority = Authority()
pair = authority.seed_grant()
authority.clock.advance(40)
try:
    authority.refresh(pair["refresh_token"], CLIENT_ID)
except OAuthError as error:
    print("恰好空闲40秒：", error.error)
print("拒绝没有签发新代：", authority.stats["refresh_successes"] == 0)

恰好空闲40秒： invalid_grant
拒绝没有签发新代： True


## 5．真正通过 HTTP 跑一遍生命周期

两个监听器都仅绑定回环地址：

| 服务 | 路径 | 用途 |
|---|---|---|
| AS | `POST /token` | form 格式刷新，返回新的 pair |
| AS | `POST /revoke` | 撤销已有或未知凭证，成功为空正文200 |
| RS | `GET /docs` | 实时核验后的只读资料 |
| RS | `GET /calculate` | 需要 `calc:use` 的固定教学计算 |
| RS | `GET /cached/docs` | 故意启用5秒授权元数据缓存，用于后面的反例 |

不提供网页登录或网络初次签发入口。token 响应使用 `Cache-Control: no-store`；没有请求正文或 Authorization 日志。

In [13]:
def demo_lifecycle() -> dict:
    with LocalServers() as servers:
        issued = servers.authority.seed_grant()
        manager = TokenManager(servers, issued)
        before = manager.call_readonly()
        servers.authority.clock.advance(10)
        expired_status, _, _ = request(servers.rs_origin, "GET", "/docs", bearer=issued["access_token"])
        after = manager.call_readonly()
        status_before_revoke = manager.summary()
        revoke_status = manager.revoke_and_forget()
        after_revoke, _, _ = request(servers.rs_origin, "GET", "/docs", bearer=issued["access_token"])
        return {"initial_resource_status": before["status"],
                "expired_old_access_status": expired_status,
                "refreshed_resource_status": after["status"],
                "client_before_revoke": status_before_revoke,
                "logout": revoke_status, "old_access_after_revoke": after_revoke,
                "authority_stats": copy.deepcopy(servers.authority.stats), "http_trace": list(servers.trace)}

In [14]:
lifecycle_report = demo_lifecycle()
print(json.dumps(lifecycle_report, ensure_ascii=False, indent=2))

{
  "initial_resource_status": 200,
  "expired_old_access_status": 401,
  "refreshed_resource_status": 200,
  "client_before_revoke": {
    "state": "ready",
    "refresh_requests": 1,
    "has_local_credentials": true,
    "access_seconds_left": 10.0
  },
  "logout": "revocation_confirmed",
  "old_access_after_revoke": 401,
  "authority_stats": {
    "refresh_attempts": 1,
    "refresh_successes": 1,
    "reuse_detected": 0,
    "revocation_requests": 1
  },
  "http_trace": [
    {
      "server": "rs",
      "method": "GET",
      "path": "/docs",
      "status": 200
    },
    {
      "server": "rs",
      "method": "GET",
      "path": "/docs",
      "status": 401
    },
    {
      "server": "as",
      "method": "POST",
      "path": "/token",
      "status": 200
    },
    {
      "server": "rs",
      "method": "GET",
      "path": "/docs",
      "status": 200
    },
    {
      "server": "as",
      "method": "POST",
      "path": "/revoke",
      "status": 200
    },
    {
  

## 6．并发刷新：两个工作线程都可能认为旧令牌还可以用

错误结构是：每个工具请求分别读取同一份旧 refresh，再自行刷新。

```text
线程 A：RT₀ → 成功获得 RT₁
线程 B：RT₀ → 旧值重用 → 撤销家族
```

因此，**收到200的线程也不能保证返回的 AT₁ 到稍后仍然有效**。

本课的修复是 single-flight：同一进程共享一个 `TokenManager`，把“再次检查到期 → HTTP刷新 → 替换完整pair”放在同一个锁内。

第二个线程进入锁以后看到第一线程的新 AT，会直接复用，而不是继续发送旧 RT。

In [15]:
def demo_concurrency() -> dict:
    # 故意构造两个没有共享刷新管理器的请求。谁先完成不作断言。
    with LocalServers() as servers:
        issued = servers.authority.seed_grant()
        barrier = threading.Barrier(2)
        def raw_attempt(_):
            barrier.wait(timeout=5)
            return refresh_http(servers, issued["refresh_token"])
        with ThreadPoolExecutor(max_workers=2) as pool:
            results = list(pool.map(raw_attempt, range(2)))
        winner = next(body for status, body, _ in results if status == 200)
        state_after_race = servers.authority.inspect_access(winner["access_token"], servers.authority.resource)
        race = {"statuses": sorted(x[0] for x in results),
                "successes": servers.authority.stats["refresh_successes"],
                "reuse_detected": servers.authority.stats["reuse_detected"],
                "winner_access_active_after_both_finish": state_after_race["active"]}
    with LocalServers() as servers:
        manager = TokenManager(servers, servers.authority.seed_grant())
        servers.authority.clock.advance(10)
        barrier = threading.Barrier(6)
        def guarded(_):
            barrier.wait(timeout=5)
            return manager.get_access_token()
        with ThreadPoolExecutor(max_workers=6) as pool:
            values = list(pool.map(guarded, range(6)))
        guarded_result = {"callers": 6, "distinct_access_tokens": len(set(values)),
                          "refresh_requests": manager.refresh_requests,
                          "refresh_successes": servers.authority.stats["refresh_successes"],
                          "reuse_detected": servers.authority.stats["reuse_detected"]}
    return {"uncoordinated": race, "single_flight": guarded_result,
            "boundary": "只验证单进程共享管理器；不是分布式 single-flight"}

In [16]:
concurrency_report = demo_concurrency()
print(json.dumps(concurrency_report, ensure_ascii=False, indent=2))

{
  "uncoordinated": {
    "statuses": [
      200,
      400
    ],
    "successes": 1,
    "reuse_detected": 1,
    "winner_access_active_after_both_finish": false
  },
  "single_flight": {
    "callers": 6,
    "distinct_access_tokens": 1,
    "refresh_requests": 1,
    "refresh_successes": 1,
    "reuse_detected": 0
  },
  "boundary": "只验证单进程共享管理器；不是分布式 single-flight"
}


预期观察：无协调的两次请求返回 `[200,400]`，家族触发一次重用；共享管理器的六次请求只产生一次刷新，得到同一份 AT。

**一个线程锁只覆盖一个管理器对象所在的进程。** 两个程序、两台机器、重启后的旧副本，仍可能竞争。跨进程需要可信凭证服务、事务/CAS和刷新状态管理；本课没有实现它们。

资源请求可能在发出后才过期。当前客户端会把401记录为重新核对授权，不自动重放工具；特别是写操作，不能因刷新了凭证就无条件重试。

## 7．最难的窗口：刷新已提交，响应丢失

本课真的让服务端先轮换，再关闭 TCP 连接，不发送响应：

```text
客户端还持有 RT₀
AS 已经消费 RT₀，签发 RT₁
客户端没有收到 RT₁
```

安全的课堂策略：状态变为 `refresh_unknown`，后续调用直接停止；核对或重新授权。**不自动重发 RT₀。**

这不是声明所有服务都不能提供恢复能力。某些服务有专门的幂等刷新或宽限契约，但必须了解并验证该契约。本课没有实现，也不假设它存在。

In [17]:
def demo_response_loss() -> dict:
    with LocalServers() as servers:
        issued = servers.authority.seed_grant()
        manager = TokenManager(servers, issued)
        servers.authority.clock.advance(10)
        servers.arm_response_loss()  # 服务器先提交轮换，再真实关闭 socket 不发响应
        states = []
        for _ in range(2):
            try:
                manager.get_access_token()
            except ReauthorizationRequired as e:
                states.append(str(e))
        safe = {"client_states": states, "network_refresh_requests": manager.refresh_requests,
                "server_committed_rotations": servers.authority.stats["refresh_successes"],
                "reuse_detected": servers.authority.stats["reuse_detected"],
                "automatic_retry": False}
        # 可信实验代码故意做错误重试，不能作为客户端推荐行为。
        status, body, _ = refresh_http(servers, issued["refresh_token"])
        unsafe = {"retry_status": status, "error": body["error"],
                  "family_revoked": servers.authority.family_status(issued["refresh_token"])["revoked"]}
    return {"safe_client": safe, "deliberate_bad_retry": unsafe,
            "note": "没有实现服务专属的刷新重试宽限或持久凭证恢复；回执未知要求核对/重新授权"}

In [18]:
lost_report = demo_response_loss()
print(json.dumps(lost_report, ensure_ascii=False, indent=2))

{
  "safe_client": {
    "client_states": [
      "refresh_unknown",
      "refresh_unknown"
    ],
    "network_refresh_requests": 1,
    "server_committed_rotations": 1,
    "reuse_detected": 0,
    "automatic_retry": false
  },
  "deliberate_bad_retry": {
    "retry_status": 400,
    "error": "invalid_grant",
    "family_revoked": true
  },
  "note": "没有实现服务专属的刷新重试宽限或持久凭证恢复；回执未知要求核对/重新授权"
}


实验分两段：前半段是客户端推荐行为；后半段由可信测试代码**故意发送错误重试**，验证它触发家族撤销。

两个片段不能拼成“客户端自动先停一下，再照旧重试”。看到 `invalid_grant` 后，没有无限刷新循环。

整对凭证的替换在内存锁内完成，防止出现“新 AT 配旧 RT”。磁盘写入、崩溃恢复和加密存储没有实现；AS 与客户端之间也没有跨机器原子事务。

## 8．退出登录、撤销 access、撤销 refresh

本地删除：只清除这个客户端手上的引用；复制到其他地方的 AT 还可能有效。

本课撤销策略：

| 操作 | 效果 |
|---|---|
| `logout_local()` | 不发送网络请求，不能承诺服务器已撤销 |
| 撤销一个 AT | 该 AT 失效，RT 仍可刷新 |
| 撤销 RT | 家族失效，实时验证拒绝所有相关 AT |
| 撤销不存在的 token | 返回200，正文为空 |

未知 token 也返回200，是 RFC7009 规定的行为；**200不能证明某个 token 之前一定有效**。无效的 `token_type_hint` 只是提示，不能改变真实 token 的识别。

撤销不同类型时是否级联，由服务策略决定；本表不是统一标准。[RFC7009 §2](https://www.rfc-editor.org/rfc/rfc7009.html#section-2)

In [19]:
def demo_revocation() -> dict:
    with LocalServers() as servers:
        issued = servers.authority.seed_grant()
        manager = TokenManager(servers, issued)
        copied_access = issued["access_token"]
        manager.logout_local()
        still_works, _, _ = request(servers.rs_origin, "GET", "/docs", bearer=copied_access)
        revoked, empty, _ = revoke_http(servers, issued["refresh_token"])
        blocked, _, _ = request(servers.rs_origin, "GET", "/docs", bearer=copied_access)
        unknown, unknown_body, _ = revoke_http(servers, "unknown-token-value")
        at_only = servers.authority.seed_grant()
        revoke_http(servers, at_only["access_token"], "access_token")
        refresh_status, _, _ = refresh_http(servers, at_only["refresh_token"])
        return {"copied_access_after_local_logout": still_works,
                "revoke_refresh_status": revoked, "revoke_response_empty": empty == {},
                "copied_access_after_family_revoke": blocked,
                "unknown_token_revoke_status": unknown,
                "unknown_revoke_response_empty": unknown_body == {},
                "refresh_after_access_only_revocation": refresh_status,
                "cascade_policy": "撤销RT影响家族；只撤销AT不撤销RT。这是本课明确选择。"}

In [20]:
revocation_report = demo_revocation()
print(json.dumps(revocation_report, ensure_ascii=False, indent=2))

{
  "copied_access_after_local_logout": 200,
  "revoke_refresh_status": 200,
  "revoke_response_empty": true,
  "copied_access_after_family_revoke": 401,
  "unknown_token_revoke_status": 200,
  "unknown_revoke_response_empty": true,
  "refresh_after_access_only_revocation": 200,
  "cascade_policy": "撤销RT影响家族；只撤销AT不撤销RT。这是本课明确选择。"
}


## 9．撤销传播与缓存：同一个令牌，两条路径给出不同结果

实时 RS 每次查看权威状态；缓存 RS 暂时复用原来的 `active=True`。

因此可能出现：

```text
AS 已撤销，实时接口401；
缓存接口暂时200；缓存到期后才重新验证，变为401。
```

缓存有效期必须限制在 access 的 `exp` 之前。**不能为了少访问AS，让已经过期的令牌继续通过。**

这对应 RFC7662 明确讨论的性能与撤销新鲜度取舍。本课使用可信内存查询模拟该语义，没有冒充实现了完整网络 introspection。

[参考：RFC7662 §4](https://www.rfc-editor.org/rfc/rfc7662.html#section-4)

In [21]:
def demo_cache() -> dict:
    with LocalServers() as servers:
        issued = servers.authority.seed_grant()
        before, _, _ = request(servers.rs_origin, "GET", "/cached/docs", bearer=issued["access_token"])
        revoke_http(servers, issued["refresh_token"])
        immediate, _, _ = request(servers.rs_origin, "GET", "/docs", bearer=issued["access_token"])
        stale, _, _ = request(servers.rs_origin, "GET", "/cached/docs", bearer=issued["access_token"])
        servers.authority.clock.advance(5)
        after_ttl, _, _ = request(servers.rs_origin, "GET", "/cached/docs", bearer=issued["access_token"])
        return {"before": before, "live_after_revoke": immediate,
                "cached_before_ttl": stale, "cached_at_ttl": after_ttl,
                "cache_seconds": 5, "time_kind": "受控时钟，不是实测5秒性能"}

In [22]:
cache_report = demo_cache()
print(json.dumps(cache_report, ensure_ascii=False, indent=2))

{
  "before": 200,
  "live_after_revoke": 401,
  "cached_before_ttl": 200,
  "cached_at_ttl": 401,
  "cache_seconds": 5,
  "time_kind": "受控时钟，不是实测5秒性能"
}


## 10．放回 MCP / Agent：凭证属于宿主，不属于模型

模型可以提出 `read_doc` 或 `calculate`，但宿主根据固定的工具来源与用户授权选用凭证。

不要把 access/refresh token 加入聊天上下文、检索文档、模型可读日志或普通工具返回内容。也不要根据工具返回的任意 URL 自动转发 Bearer。

| 结果 | 宿主需要做的事 |
|---|---|
| 本地已知 AT 即将到期 | 受控地single-flight刷新 |
| RS返回403 | 权限问题，不循环刷新 |
| `/token`返回invalid_grant | 重新核对或重新授权 |
| 刷新无可靠响应 | 记录未知，不盲目重用旧RT |
| 刷新成功 | 仍需原来的白名单、对象权限和高风险操作审批 |

普通 Bearer 被盗后可被使用；本课轮换不等于 DPoP、mTLS或无条件抗重放。[RFC6750](https://www.rfc-editor.org/rfc/rfc6750.html)

In [23]:
with LocalServers() as demo_servers:
    read_only = demo_servers.authority.seed_grant("docs:read")
    manager = TokenManager(demo_servers, read_only)
    denied = manager.call_readonly("/calculate")
    print("计算接口状态：", denied["status"])
    print("因此自动刷新了几次：", manager.refresh_requests)
    print("计算实际执行次数：", demo_servers.executions["calculate"])

计算接口状态： 403
因此自动刷新了几次： 0
计算实际执行次数： 0


## 11．自动检查

以下是完整检查定义。没有 `assert` 优化开关绕过的问题：检查器显式在失败时抛异常。

覆盖类型解析、scope、期限、轮换、家族撤销、并发、真实HTTP丢回执、缓存和客户端停止状态。**不是完整OAuth测试套件或生产安全评估。**

In [24]:
class Checks:
    def __init__(self):
        self.names = []
    def ok(self, name: str, condition: bool):
        if not condition:
            raise AssertionError(name)
        self.names.append(name)
    def error(self, name: str, fn, kind=OAuthError, code=None):
        try:
            fn()
        except kind as e:
            if code is not None and getattr(e, "error", None) != code:
                raise AssertionError(name + ": error code") from e
            self.names.append(name)
            return
        raise AssertionError(name + ": expected exception")


def run_checks() -> dict:
    t = Checks()
    t.ok("scope_parse", parse_scope("docs:read calc:use") == ALLOWED_SCOPES)
    t.error("scope_duplicate", lambda: parse_scope("docs:read docs:read"), LessonError)
    t.error("scope_unknown", lambda: parse_scope("admin"), LessonError)
    t.error("duplicate_form", lambda: strict_form("a=1&a=2"), LessonError)
    t.error("bad_percent", lambda: strict_form("a=%QZ"), LessonError)
    t.error("duplicate_json", lambda: strict_json('{"x":1,"x":2}'), LessonError)
    t.error("nan_json", lambda: strict_json('{"x":NaN}'), LessonError)
    t.error("clock_backwards", lambda: ManualClock().advance(-1), LessonError)
    t.error("clock_nan", lambda: ManualClock().advance(float("nan")), LessonError)
    t.error("bad_ttl", lambda: Policy(access_ttl=0), LessonError)
    a = Authority(); p = a.seed_grant()
    t.ok("two_distinct_tokens", p["access_token"] != p["refresh_token"])
    t.ok("only_digests_in_server_records", p["access_token"] not in a.accesses and
         p["refresh_token"] not in a.refreshes and token_digest(p["refresh_token"]) in a.refreshes)
    t.ok("initial_access_active", a.inspect_access(p["access_token"], a.resource)["active"])
    t.ok("refresh_not_access", not a.inspect_access(p["refresh_token"], a.resource)["active"])
    t.ok("wrong_audience", not a.inspect_access(p["access_token"], "wrong")["active"])
    t.error("access_not_refresh", lambda: a.refresh(p["access_token"], CLIENT_ID), code="invalid_grant")
    t.error("client_binding", lambda: a.refresh(p["refresh_token"], "other"), code="invalid_grant")
    t.ok("wrong_client_not_consume", not a.refreshes[token_digest(p["refresh_token"])].used)
    n = a.refresh(p["refresh_token"], CLIENT_ID)
    t.ok("tokens_rotate", n["refresh_token"] != p["refresh_token"] and n["access_token"] != p["access_token"])
    t.ok("old_refresh_marked_used", a.refreshes[token_digest(p["refresh_token"])].used)
    t.ok("old_access_survives_normal_rotation", a.inspect_access(p["access_token"], a.resource)["active"])
    t.ok("generation_advances", a.family_status(n["refresh_token"])["generation"] == 1)
    t.error("reuse_denied", lambda: a.refresh(p["refresh_token"], CLIENT_ID), code="invalid_grant")
    t.ok("reuse_cascades_current_access", not a.inspect_access(n["access_token"], a.resource)["active"])
    t.error("reuse_cascades_current_refresh", lambda: a.refresh(n["refresh_token"], CLIENT_ID), code="invalid_grant")
    a = Authority(); p = a.seed_grant("docs:read")
    t.error("scope_escalation", lambda: a.refresh(p["refresh_token"], CLIENT_ID, "calc:use"), code="invalid_scope")
    t.error("resource_switch", lambda: a.refresh(p["refresh_token"], CLIENT_ID, resource="elsewhere"), code="invalid_target")
    t.ok("validation_failure_not_consume", not a.refreshes[token_digest(p["refresh_token"])].used)
    a.clock.advance(10)
    t.ok("exact_access_expiry", not a.inspect_access(p["access_token"], a.resource)["active"])
    a.refresh(p["refresh_token"], CLIENT_ID)
    t.ok("expired_access_does_not_expire_refresh", a.stats["refresh_successes"] == 1)
    a = Authority(); p = a.seed_grant(); a.clock.advance(40)
    t.error("idle_boundary", lambda: a.refresh(p["refresh_token"], CLIENT_ID), code="invalid_grant")
    a = Authority(); p = a.seed_grant(); a.revoke(p["access_token"], CLIENT_ID)
    t.ok("access_only_revoked", not a.inspect_access(p["access_token"], a.resource)["active"])
    a.refresh(p["refresh_token"], CLIENT_ID)
    t.ok("access_revoke_leaves_refresh", a.stats["refresh_successes"] == 1)
    a = Authority(); p = a.seed_grant(); a.revoke(p["refresh_token"], "other")
    t.ok("wrong_client_no_revoke", a.inspect_access(p["access_token"], a.resource)["active"])
    a.revoke(p["refresh_token"], CLIENT_ID, "not_a_real_hint")
    t.ok("invalid_hint_ignored", not a.inspect_access(p["access_token"], a.resource)["active"])
    a.revoke("unknown", CLIENT_ID)
    t.ok("unknown_revoke_success", a.stats["revocation_requests"] == 3)
    a = Authority(); p = a.seed_grant(); q = a.seed_grant()
    a.revoke(p["refresh_token"], CLIENT_ID)
    t.ok("other_family_untouched", a.inspect_access(q["access_token"], a.resource)["active"])
    a = Authority(policy=Policy(access_ttl=2)); p = a.seed_grant()
    g = ResourceGuard(a, 999); g.authorize(p["access_token"], "docs:read")
    a.clock.advance(2)
    t.error("cache_cannot_outlive_access", lambda: g.authorize(p["access_token"], "docs:read"), code="invalid_token")
    with LocalServers() as s:
        p = s.authority.seed_grant("docs:read")
        status, _, headers = request(s.rs_origin, "GET", "/docs")
        t.ok("http_missing_bearer", status == 401 and "www-authenticate" in headers)
        status, _, _ = request(s.rs_origin, "GET", "/docs", bearer=p["refresh_token"])
        t.ok("http_refresh_denied_at_resource", status == 401)
        status, _, _ = request(s.rs_origin, "GET", "/docs", bearer=p["access_token"])
        t.ok("http_access_success", status == 200)
        status, body, _ = request(s.rs_origin, "GET", "/calculate", bearer=p["access_token"])
        t.ok("http_scope_denied", status == 403 and body["error"] == "insufficient_scope")
        t.ok("denied_tool_not_executed", s.executions["calculate"] == 0)
        manager = TokenManager(s, p)
        t.ok("no_refresh_for_403", manager.call_readonly("/calculate")["status"] == 403
             and manager.refresh_requests == 0)
        status, _, _ = request(s.as_origin, "POST", "/token", body_override=b"grant_type=refresh_token&grant_type=x")
        t.ok("http_duplicate_fields", status == 400)
        status, _, _ = request(s.as_origin, "POST", "/token", form={"grant_type": "authorization_code"})
        t.ok("http_unsupported_grant", status == 400)
        status, _, _ = request(s.as_origin, "POST", "/token", form={"grant_type": "refresh_token"})
        t.ok("http_missing_refresh", status == 400)
        status, _, _ = request(s.rs_origin, "GET", "/docs", bearer=p["access_token"],
                               extra_headers={"Origin": "https://evil.invalid"})
        t.ok("browser_origin_rejected", status == 403)
        status, new, headers = refresh_http(s, p["refresh_token"])
        t.ok("token_response_no_store", status == 200 and headers.get("cache-control") == "no-store")
        status, empty, _ = revoke_http(s, new["refresh_token"])
        t.ok("revoke_empty_200", status == 200 and empty == {})
        status, empty, _ = revoke_http(s, "unknown-token-value")
        t.ok("unknown_revoke_empty_200", status == 200 and empty == {})
        encoded = canonical(s.trace)
        t.ok("transport_log_no_credentials", all(v not in encoded for v in
                (p["access_token"], p["refresh_token"], new["access_token"], new["refresh_token"])))
        t.error("reject_remote_origin", lambda: request("https://example.com", "GET", "/docs"), LessonError)
        # 在同一进程加锁，避免出现半份本地 token 响应。
        t.ok("pair_repr_hides_tokens", p["access_token"] not in repr(manager.pair)
             and p["refresh_token"] not in repr(manager.pair))
    lifecycle = demo_lifecycle()
    t.ok("lifecycle_expired_401", lifecycle["expired_old_access_status"] == 401)
    t.ok("lifecycle_single_refresh", lifecycle["client_before_revoke"]["refresh_requests"] == 1)
    t.ok("lifecycle_after_refresh_200", lifecycle["refreshed_resource_status"] == 200)
    concurrent = demo_concurrency()
    t.ok("race_one_issue_one_error", concurrent["uncoordinated"]["statuses"] == [200, 400])
    t.ok("race_winner_revoked", not concurrent["uncoordinated"]["winner_access_active_after_both_finish"])
    t.ok("singleflight_one_http", concurrent["single_flight"]["refresh_requests"] == 1)
    t.ok("singleflight_same_access", concurrent["single_flight"]["distinct_access_tokens"] == 1)
    t.ok("singleflight_no_reuse", concurrent["single_flight"]["reuse_detected"] == 0)
    lost = demo_response_loss()
    t.ok("lost_response_committed", lost["safe_client"]["server_committed_rotations"] == 1)
    t.ok("lost_response_no_retry", lost["safe_client"]["network_refresh_requests"] == 1)
    t.ok("lost_response_unknown", lost["safe_client"]["client_states"] == ["refresh_unknown"] * 2)
    t.ok("bad_retry_revoke", lost["deliberate_bad_retry"]["family_revoked"])
    revoke = demo_revocation()
    t.ok("local_delete_not_revoke", revoke["copied_access_after_local_logout"] == 200)
    t.ok("server_revoke_effective", revoke["copied_access_after_family_revoke"] == 401)
    t.ok("access_revoke_not_logout", revoke["refresh_after_access_only_revocation"] == 200)
    cache = demo_cache()
    t.ok("cached_result_can_be_stale", cache["live_after_revoke"] == 401 and cache["cached_before_ttl"] == 200)
    t.ok("cache_eventually_rechecks", cache["cached_at_ttl"] == 401)
    details = demo_scope_and_expiry()
    t.ok("access_narrowing", details["scopes"]["narrow_read"] == 200 and details["scopes"]["narrow_calc"] == 403)
    t.ok("rt_retains_grant", details["scopes"]["restore_original_scope_calc"] == 200)
    t.ok("original_scope_ceiling", details["scopes"]["beyond_grant_error"] == "invalid_scope")
    t.ok("fixed_absolute_deadline", details["absolute_remaining_after_rotations"] == [45, 31, 17, 3])
    t.ok("absolute_expiry_enforced", details["at_absolute_boundary"] == "invalid_grant")
    t.ok("refresh_expiry_not_automatic_access_revoke", details["access_can_outlive_refresh_eligibility"])
    with LocalServers() as s:
        p = s.authority.seed_grant(); m = TokenManager(s, p)
        s.authority.revoke(p["refresh_token"], CLIENT_ID)
        t.ok("revoked_resource_401", m.call_readonly()["status"] == 401)
        t.error("401_no_refresh_loop", m.get_access_token, ReauthorizationRequired)
        t.ok("401_keeps_no_refresh_attempt", m.refresh_requests == 0)
        p = s.authority.seed_grant(); m = TokenManager(s, p)
        s.authority.clock.advance(40)
        t.error("invalid_grant_reauth", m.get_access_token, ReauthorizationRequired)
        t.ok("invalid_grant_state", m.state == "reauth_required")
        t.error("invalid_grant_second_no_network", m.get_access_token, ReauthorizationRequired)
        t.ok("invalid_grant_once", m.refresh_requests == 1)
    return {"passed": len(t.names), "names": t.names,
            "scope": "标准库与真实回环HTTP；没有真实身份、TLS、模型或生产安全认证"}

In [25]:
check_report = run_checks()
print("固定检查通过：", check_report["passed"], "项")
print(check_report["scope"])

固定检查通过： 84 项
标准库与真实回环HTTP；没有真实身份、TLS、模型或生产安全认证


## 12．保存脱敏报告

每次保存到新的时间戳目录。报告只包含状态、计数、相对时间和不含凭证的HTTP路径。

初始授权、当前有效凭证和服务端索引都留在进程内；退出后不可恢复。**这不等于安全秘密存储设计，也不代表Python内存能可靠擦除。**

终端完整脚本还会额外运行五种故障注入，修改的是课程自己的可信源码，不接受用户任意代码。

In [26]:
def save_reports(root: str | Path, reports: dict) -> Path:
    root = Path(root)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    out = root / ("run_" + stamp)
    out.mkdir(parents=True, exist_ok=False)
    for name, value in reports.items():
        (out / (name + ".json")).write_text(
            json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return out

In [27]:
experiment_report = {
    "lifecycle": lifecycle_report,
    "concurrency": concurrency_report,
    "lost-response": lost_report,
    "revocation": revocation_report,
    "cache": cache_report,
    "scope-expiry": scope_expiry_report,
}
output_dir = save_reports("lesson_54_outputs", {
    "experiment": experiment_report,
    "checks": check_report,
    "manifest": {
        "lesson": 54, "version": VERSION, "mode": "notebook_loopback_http",
        "python": platform.python_version(), "model_used": False,
        "tls": False, "real_user_authentication": False,
        "clock": "host-controlled logical clock", "sources": SOURCES,
    },
})
print("脱敏报告目录：", output_dir.resolve())

脱敏报告目录： /mnt/data/_lesson54_validation/notebook/lesson_54_outputs/run_20261102T133548_265672Z


# 独立练习

先完成下面三个练习，再展开后面的参考实现。检查开关默认关闭，因此**运行全部单元，不等于完成练习**。

## 练习 A：本次 access scope

实现 `access_scopes_ex(granted, requested)`：`requested=None` 时返回原批准范围；有请求时只接受原范围的子集。拒绝超范围，用 `OAuthError("invalid_scope")`，不改变输入。

这里只计算本次 AT 的权限，不改变下一代 RT 的原始 grant。

In [28]:
def access_scopes_ex(granted: frozenset[str], requested: str | None):
    raise NotImplementedError("完成 scope 上限检查")


def check_scope_answer(fn):
    grant = frozenset({"docs:read", "calc:use"})
    checks = Checks()
    checks.ok("默认原权限", fn(grant, None) == grant)
    checks.ok("本次缩小", fn(grant, "docs:read") == frozenset({"docs:read"}))
    checks.error("未授予的calc", lambda: fn(frozenset({"docs:read"}), "calc:use"), code="invalid_scope")
    checks.error("重复scope", lambda: fn(grant, "docs:read docs:read"), code="invalid_scope")
    checks.error("未知scope", lambda: fn(grant, "admin"), code="invalid_scope")
    checks.ok("原grant不变", grant == frozenset({"docs:read", "calc:use"}))
    return len(checks.names)

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print("练习A：", check_scope_answer(access_scopes_ex), "项通过")
else:
    print("练习A待完成")

练习A待完成


## 练习 B：授权缓存截止时间

实现 `cache_deadline_ex(now, ttl, access_exp)`。缓存期限取 `min(now+ttl, access_exp)`。

三个参数必须是有限数字而不是布尔值；`ttl` 不得为负；AT已经到期时拒绝缓存。返回绝对截止时间。

该函数不负责检查撤销通知，它只保证缓存不会越过本来已经知道的到期时刻。

In [29]:
def cache_deadline_ex(now, ttl, access_exp):
    raise NotImplementedError("实现缓存截止计算")


def check_cache_answer(fn):
    checks = Checks()
    checks.ok("ttl更短", fn(100, 5, 120) == 105)
    checks.ok("AT更早到期", fn(100, 50, 120) == 120)
    checks.ok("零缓存", fn(100, 0, 120) == 100)
    checks.error("已到期", lambda: fn(100, 5, 100), LessonError)
    checks.error("负ttl", lambda: fn(100, -1, 120), LessonError)
    checks.error("非有限", lambda: fn(100, float("inf"), 120), LessonError)
    checks.error("布尔伪数字", lambda: fn(True, 5, 120), LessonError)
    return len(checks.names)

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print("练习B：", check_cache_answer(cache_deadline_ex), "项通过")
else:
    print("练习B待完成")

练习B待完成


## 练习 C：单次刷新与整对替换

实现 `get_access_ex(manager)`。使用已有 `manager.lock`、`manager.pair`、`refresh_http()` 与 `_decode_pair()` 等基础组件，但**不要直接调用现成的 `get_access_token()`**。

检查目标：

1. 未到期时不刷新；六个并发请求到期时只刷新一次。
2. 收到 `invalid_grant` 进入 `reauth_required`；未知传输结果进入 `refresh_unknown`。
3. 刷新响应验证完整以后才替换pair；异常后不自动再次发请求。

不要把等待者的锁拿到HTTP之前又释放，那样仍然会造成并发刷新。

In [30]:
def get_access_ex(manager: TokenManager) -> str:
    raise NotImplementedError("完成单进程single-flight")


def check_singleflight_answer(fn):
    checks = Checks()
    with LocalServers() as servers:
        manager = TokenManager(servers, servers.authority.seed_grant())
        checks.ok("未到期复用", fn(manager) == manager.pair.access_token and manager.refresh_requests == 0)
        servers.authority.clock.advance(10)
        barrier = threading.Barrier(6)
        def invoke(_):
            barrier.wait(timeout=5)
            return fn(manager)
        with ThreadPoolExecutor(max_workers=6) as pool:
            tokens = list(pool.map(invoke, range(6)))
        checks.ok("只刷新一次", manager.refresh_requests == 1)
        checks.ok("共享新AT", len(set(tokens)) == 1)
        checks.ok("新RT仍可使用", not servers.authority.family_status(manager.pair.refresh_token)["refresh_used"])
    with LocalServers() as servers:
        manager = TokenManager(servers, servers.authority.seed_grant())
        servers.authority.clock.advance(10)
        servers.arm_response_loss()
        checks.error("回执未知停止", lambda: fn(manager), ReauthorizationRequired)
        checks.ok("未知状态", manager.state == "refresh_unknown")
        checks.error("不得自动再请求", lambda: fn(manager), ReauthorizationRequired)
        checks.ok("请求仅一次", manager.refresh_requests == 1)
    return len(checks.names)

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print("练习C：", check_singleflight_answer(get_access_ex), "项通过")
else:
    print("练习C待完成")

练习C待完成


---
# 参考答案

请先独立完成练习。以下函数使用不同名称，不覆盖你的实现。检查只覆盖指定行为，不证明全部安全性质。

In [31]:
def access_scopes_ref(granted: frozenset[str], requested: str | None):
    try:
        selected = granted if requested is None else parse_scope(requested)
    except LessonError:
        raise OAuthError("invalid_scope") from None
    if not selected <= granted:
        raise OAuthError("invalid_scope")
    return frozenset(selected)

print("参考A：", check_scope_answer(access_scopes_ref), "项通过")

参考A： 6 项通过


In [32]:
def cache_deadline_ref(now, ttl, access_exp):
    require(all(type(value) in (int, float) and math.isfinite(value)
                for value in (now, ttl, access_exp)), "参数必须是有限数字")
    require(ttl >= 0 and access_exp > now, "不能缓存已过期令牌或使用负ttl")
    return min(now + ttl, access_exp)

print("参考B：", check_cache_answer(cache_deadline_ref), "项通过")

参考B： 7 项通过


In [33]:
def get_access_ref(manager: TokenManager) -> str:
    with manager.lock:
        if manager.state != "ready" or manager.pair is None:
            raise ReauthorizationRequired(manager.state)
        if manager.clock.now() < manager.pair.expires_at - manager.leeway:
            return manager.pair.access_token
        manager.state = "refreshing"
        manager.refresh_requests += 1
        request_started = manager.clock.now()
        try:
            status, body, _ = refresh_http(manager.servers, manager.pair.refresh_token)
            if status != 200:
                manager.state = "reauth_required" if body.get("error") == "invalid_grant" else "refresh_failed"
                raise ReauthorizationRequired(manager.state)
            new_pair = manager._decode_pair(body, request_started)
        except (TransportUnknown, LessonError):
            manager.state = "refresh_unknown"
            raise ReauthorizationRequired(manager.state) from None
        manager.pair = new_pair
        manager.state = "ready"
        return new_pair.access_token

print("参考C：", check_singleflight_answer(get_access_ref), "项通过")

参考C： 8 项通过


## 终端使用与本课验收

完整运行：

```bash
python lesson_54_token_lifecycle_rotation_and_revocation.py
```

只检查：

```bash
python lesson_54_token_lifecycle_rotation_and_revocation.py --checks-only
```

只观察并发或回执丢失：

```bash
python lesson_54_token_lifecycle_rotation_and_revocation.py --case concurrency
python lesson_54_token_lifecycle_rotation_and_revocation.py --case lost-response
```

本机交互练习（不会打印凭证）：

```bash
python lesson_54_token_lifecycle_rotation_and_revocation.py --interactive
```

依次尝试 `status`、`read`、`calc`、`advance 10`、`read`、`revoke`、`read`、`authorize`、`quit`。

`authorize` 只重新生成模拟批准，不是真实浏览器授权。别把这个夹具作为真实自动审批。

**验收问题：** 为什么403不应无限刷新？为什么两个200/400并发结果中，200返回的令牌也可能失效？为什么撤销refresh与仅删除本地文件不一样？为什么收到EOF以后不直接重发旧refresh？

## 边界与下一步

本课验证的是受控的单进程授权生命周期：有真实回环HTTP、真实多线程竞争和真实socket响应丢失；没有公网、TLS、账号登录、JWT、DPoP、跨进程凭证锁或数据库恢复。

没有将标准库 `http.server` 作为生产部署建议。时间是逻辑时钟，所有实验数据不是延迟/吞吐量结果。未知错误从严停止，不能把这当作生产可用性最优策略。

**下一课：第55课——推理任务与可验证评分器。** 回到模型能力，建立数学与代码任务的确定评分规则，再研究多次采样和测试时计算，区分答案变好与只是多用了计算预算。

## 原始资料

- [RFC6749 §6：Refresh grant与scope约束](https://www.rfc-editor.org/rfc/rfc6749.html#section-6)
- [RFC9700 §4.14：Refresh token保护、轮换与重用检测](https://www.rfc-editor.org/rfc/rfc9700.html#section-4.14)
- [RFC7009：令牌撤销、未知token与级联策略](https://www.rfc-editor.org/rfc/rfc7009.html)
- [RFC7662 §4：授权元数据缓存与撤销窗口](https://www.rfc-editor.org/rfc/rfc7662.html#section-4)
- [RFC6750 §3.1：invalid_token与insufficient_scope](https://www.rfc-editor.org/rfc/rfc6750.html#section-3.1)
- [Python http.server：本地教学服务的适用边界](https://docs.python.org/3/library/http.server.html)

依据：2026-11-02核对的上述标准。具体TTL、家族撤销、回执未知停机和教学scope集合，都是课程明示策略。